In [1]:
%pip install requests schedule

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import json

url = "https://api.binance.com/api/v3/ticker/24hr"

response = requests.get(url, timeout=15)
response.raise_for_status()

all_coins = response.json()

coins = [
    coin for coin in all_coins
    if coin["symbol"].endswith("USDT")
][:10]

with open("crypto_data.json", "w") as file:
    json.dump(coins, file, indent=4)

print("Data saved successfully!")

Data saved successfully!


In [3]:
def find_most_volatile_coin(data):

    coin = max(
        data,
        key=lambda x: abs(float(x["priceChangePercent"]))
    )

    return coin["symbol"]


import json

with open("crypto_data.json", "r") as file:
    data = json.load(file)

result = find_most_volatile_coin(data)

print("Most Volatile Coin:", result)

Most Volatile Coin: BNBUSDT


In [4]:
import json

with open("crypto_data.json", "r") as file:
    data = json.load(file)

prices = [float(coin["lastPrice"]) for coin in data]

average = sum(prices) / len(prices)

print("Average Price:", average)

print("\nCoins Below Average:")

for coin in data:
    if float(coin["lastPrice"]) < average:
        print(coin["symbol"], ":", coin["lastPrice"])

Average Price: 8159.797339999999

Coins Below Average:
ETHUSDT : 2249.97000000
BNBUSDT : 712.75000000
BCCUSDT : 0.00000000
NEOUSDT : 1.97600000
LTCUSDT : 56.25000000
QTUMUSDT : 0.78300000
ADAUSDT : 0.19400000
XRPUSDT : 1.26050000
EOSUSDT : 0.77990000


In [5]:
import json

def rank_coins_by_volume(data):

    sorted_coins = sorted(
        data,
        key=lambda x: float(x["quoteVolume"]),
        reverse=True
    )

    print("Top 5 Coins by Trading Volume")

    for rank, coin in enumerate(sorted_coins[:5], start=1):
        print(rank, coin["symbol"], "Volume:", coin["quoteVolume"])


with open("crypto_data.json", "r") as file:
    data = json.load(file)

rank_coins_by_volume(data)

Top 5 Coins by Trading Volume
1 BTCUSDT Volume: 2510191479.25905840
2 ETHUSDT Volume: 1245869183.98589300
3 BNBUSDT Volume: 492946839.67286000
4 XRPUSDT Volume: 251990445.34142000
5 ADAUSDT Volume: 41749750.17301000


In [ ]:
import requests
import json
import schedule
import time

def fetch_crypto_data():

    url = "https://api.binance.com/api/v3/ticker/24hr"

    for attempt in range(5):

        try:
            response = requests.get(url, timeout=15)

            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 2 ** attempt))
                print("Rate limit reached. Waiting:", wait, "seconds")
                time.sleep(wait)
                continue

            response.raise_for_status()

            all_coins = response.json()

            coins = [
                coin for coin in all_coins
                if coin["symbol"].endswith("USDT")
            ]

            coins = sorted(
                coins,
                key=lambda x: float(x["quoteVolume"]),
                reverse=True
            )[:10]

            with open("crypto_data.json", "w") as file:
                json.dump(coins, file, indent=4)

            print("Crypto data updated successfully!")

            find_most_volatile_coin(coins)
            rank_coins_by_volume(coins)

            return

        except requests.exceptions.RequestException as e:
            print("API Error:", e)

            if attempt < 4:
                wait = 2 ** attempt
                print("Retrying in", wait, "seconds...")
                time.sleep(wait)

    print("Unable to fetch data after 5 attempts.")


def find_most_volatile_coin(data):

    coin = max(
        data,
        key=lambda x: abs(float(x["priceChangePercent"]))
    )

    print("Most Volatile Coin:", coin["symbol"])


def rank_coins_by_volume(data):

    sorted_coins = sorted(
        data,
        key=lambda x: float(x["quoteVolume"]),
        reverse=True
    )

    print("\nTop 5 Coins by Volume:")

    for rank, coin in enumerate(sorted_coins[:5], start=1):
        print(rank, coin["symbol"], coin["quoteVolume"])


# Run once immediately
fetch_crypto_data()

# Schedule every hour
schedule.every(1).hours.do(fetch_crypto_data)

print("\nCrypto tracking started. Updating every hour...")

while True:
    schedule.run_pending()
    time.sleep(1)

Crypto data updated successfully!
Most Volatile Coin: BNBUSDT

Top 5 Coins by Volume:
1 BTCUSDT 2510820948.20038720
2 USDCUSDT 1914880081.93264000
3 ETHUSDT 1246515349.91333300
4 BNBUSDT 492851744.82576000
5 SOLUSDT 452367942.09928000

Crypto tracking started. Updating every hour...
